# Lecția 13 - Memoria agentului cu grafuri de cunoștințe Cognee


## Configurare

Acest notebook demonstrează cum să construiești un **asistent de codare** inteligent cu memorie persistentă folosind grafuri de cunoștințe [**Cognee**](https://www.cognee.ai/) și **Microsoft Agent Framework** (MAF).

Cognee transformă textul nestructurat într-un graf de cunoștințe structurat și interogabil, susținut de vectori încorporați — oferind agentului tău o memorie pe termen lung bogată și conștientă de relații.

### Ce vei învăța
1. **Construiește grafuri de cunoștințe**: Transformă profilurile dezvoltatorilor și cele mai bune practici în cunoștințe structurate și interogabile.
2. **Integrează Cognee cu MAF**: Folosește funcțiile `@tool` pentru a permite unui agent MAF să interogheze graful de cunoștințe Cognee.
3. **Conversații conștiente de sesiune**: Menține contextul pe parcursul mai multor întrebări în aceeași sesiune.
4. **Memorie pe termen lung**: Păstrează cunoștințe importante între sesiuni și recuperează-le în conversații noi.

### Cerințe preliminare
- Python 3.9+
- Redis rulând local (`docker run -d -p 6379:6379 redis`) pentru managementul sesiunilor
- O cheie API LLM (de ex. OpenAI) — setează `LLM_API_KEY` în `.env`
- `CACHING=true` în `.env` (necesar pentru sesiunile Cognee)
- Un proiect Microsoft Foundry cu un model de chat implementat
- `AZURE_AI_PROJECT_ENDPOINT` și `AZURE_AI_MODEL_DEPLOYMENT_NAME` în `.env`
- Azure CLI autentificat (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity "cognee[redis]==0.4.0" -q

In [ ]:
import os
from pathlib import Path
from typing import Annotated

from dotenv import load_dotenv

load_dotenv()

os.environ["LLM_API_KEY"] = os.getenv("LLM_API_KEY", "")
os.environ["CACHING"] = os.getenv("CACHING", "true")

import cognee
from cognee.modules.search.types import SearchType

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

print(f"Cognee version: {cognee.__version__}")
print(f"CACHING: {os.environ.get('CACHING')}")


In [ ]:
provider = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)

print("✅ FoundryChatClient created")


## Tipuri de Memorie ale Agentului

Această notă explorează aceleași trei tipuri de memorie din notița principală a Lecției 13, dar utilizează Cognee ca backend pentru memoria pe termen lung:

| Tip Memorie | Mecanism | Durată |
|---|---|---|
| **De lucru** | `agent.create_session()` (MAF) | O singură conversație |
| **Pe termen scurt** | Cache de sesiune Cognee (Redis) | O singură sesiune |
| **Pe termen lung** | Grafic de cunoștințe Cognee + vectori | Permanent |

### Arhitectura Memoriei Cognee
```
┌──────────────────────────┐
│      Raw Data            │  (developer profiles, docs, conversations)
└───────────┬──────────────┘
            │  cognee.add() + cognee.cognify()
            ▼
┌──────────────────────────────────────────┐
│  Knowledge Graph + Vector Embeddings     │
└───────────┬──────────────────────────────┘
            │  cognee.search()
            ▼
┌──────────────────┐       ┌────────────────┐
│  MAF Agent       │──────▶│  @tool funcs   │
│  (AgentSession)  │       │  wrapping       │
│                  │       │  cognee.search  │
└──────────────────┘       └────────────────┘
```


## Pregătiți stocarea Cognee


In [ ]:
DATA_ROOT = Path('.data_storage').resolve()
SYSTEM_ROOT = Path('.cognee_system').resolve()

DATA_ROOT.mkdir(parents=True, exist_ok=True)
SYSTEM_ROOT.mkdir(parents=True, exist_ok=True)

cognee.config.data_root_directory(str(DATA_ROOT))
cognee.config.system_root_directory(str(SYSTEM_ROOT))

await cognee.prune.prune_data()
await cognee.prune.prune_system(metadata=True)
print("✅ Cognee storage configured and reset")

## Partea 1 — Construirea bazei de cunoștințe

Ingerăm trei tipuri de date pentru a crea o bază de cunoștințe cuprinzătoare pentru asistentul nostru de programare:

1. **Profilul dezvoltatorului** — expertiza personală și backgroundul tehnic
2. **Cele mai bune practici Python** — Zen-ul Python cu ghiduri practice
3. **Conversații istorice** — sesiuni Q&A anterioare între dezvoltatori și asistenți AI


In [ ]:
developer_intro = (
    "Hi, I'm an AI/Backend engineer. "
    "I build FastAPI services with Pydantic, heavy asyncio/aiohttp pipelines, "
    "and production testing via pytest-asyncio. "
    "I've shipped low-latency APIs on AWS, Azure, and GoogleCloud."
)

python_zen_principles = """
# The Zen of Python: Practical Guide

## Key Principles With Guidance

### 1. Beautiful is better than ugly
Prefer descriptive names, clear structure, and consistent formatting.

### 2. Explicit is better than implicit
Be clear about behavior, imports, and types.

### 3. Simple is better than complex
Choose straightforward solutions first.

### 4. Flat is better than nested
Use early returns to reduce indentation.

## Modern Python Tie-ins
- Type hints reinforce explicitness
- Context managers enforce safe resource handling
- Dataclasses improve readability for data containers
"""

human_agent_conversations = """
"conversations": [
    {
      "topic": "async/await patterns",
      "user_query": "I'm building a web scraper that needs to handle thousands of URLs concurrently. What's the best way to structure this with asyncio?",
      "assistant_response": "Use asyncio with aiohttp, a semaphore to cap concurrency, TCPConnector for connection pooling, and context managers for session lifecycle."
    },
    {
      "topic": "dataclass vs pydantic",
      "user_query": "When should I use dataclasses vs Pydantic models?",
      "assistant_response": "For API input/output, prefer Pydantic: runtime validation, type coercion, JSON serialization. Integrates cleanly with FastAPI."
    },
    {
      "topic": "testing patterns",
      "user_query": "What's the best approach for pytest with async functions?",
      "assistant_response": "Use pytest-asyncio, async fixtures, and an isolated test database or mocks to reliably test async code."
    },
    {
      "topic": "error handling and logging",
      "user_query": "What's the best approach for production-ready error management?",
      "assistant_response": "Centralized error handling with custom exceptions, structured logging, and FastAPI middleware."
    }
  ]
"""

print("✅ Data sources prepared")

In [ ]:
await cognee.add(developer_intro, node_set=["developer_data"])
await cognee.add(human_agent_conversations, node_set=["developer_data"])
await cognee.add(python_zen_principles, node_set=["principles_data"])

await cognee.cognify()
print("✅ Knowledge graph built")

## Vizualizează Graful Cunoașterii

Cognee poate reda o vizualizare HTML interactivă a entităților și relațiilor pe care le-a extras.


In [ ]:
from cognee import visualize_graph

await visualize_graph('./cognee_graph.html')
print("📊 Graph saved to cognee_graph.html — open it in a browser to explore.")

## Îmbogățește memoria cu Memify

`memify()` analizează graful de cunoștințe și generează reguli inteligente — identificând tipare, cele mai bune practici și relații între concepte.


In [ ]:
await cognee.memify()
print("✅ Memory enriched with memify")

## Partea 2 — Agent MAF cu unelte Cognee

Acum creăm un agent MAF care poate interoga graful de cunoștințe Cognee prin funcțiile `@tool`. Acest lucru permite agentului să exploateze puterea completă a căutării semantice conștiente de graf, menținând în același timp contextul conversațional prin sesiuni.


In [ ]:
@tool(approval_mode="never_require")
async def search_knowledge(
    query: Annotated[str, "Natural-language question to search the knowledge graph"],
) -> str:
    """Search the Cognee knowledge graph for relevant developer knowledge, best practices, and past conversations."""
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
    )
    if not results:
        return "No relevant knowledge found."
    return str(results)


@tool(approval_mode="never_require")
async def search_principles(
    query: Annotated[str, "Question about Python principles or best practices"],
) -> str:
    """Search only the Python principles subset of the knowledge graph."""
    from cognee.modules.engine.models.node_set import NodeSet
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
        node_type=NodeSet,
        node_name=["principles_data"],
    )
    if not results:
        return "No relevant principles found."
    return str(results)


print("✅ Cognee tools defined: search_knowledge, search_principles")

In [ ]:
coding_agent = provider.as_agent(
    name="CodingAssistant",
    instructions=(
        "You are an expert coding assistant with access to a knowledge graph "
        "containing developer profiles, Python best practices, and past conversations.\n\n"
        "WORKFLOW:\n"
        "1. Use search_knowledge() to find relevant information from the full knowledge graph.\n"
        "2. Use search_principles() when the question is specifically about Python best practices.\n"
        "3. Combine retrieved knowledge with your own expertise to give comprehensive answers.\n"
        "4. Reference the developer's known tech stack (FastAPI, asyncio, Pydantic) when relevant."
    ),
)

print("✅ CodingAssistant agent created")


## Memorie de lucru cu sesiuni

`AgentSession` (creat prin `agent.create_session()`) oferă memorie de lucru într-o sesiune. Agentul poate face referire la mesaje anterioare în timp ce interoghează și graficul de cunoștințe pe termen lung al lui Cognee.


In [ ]:
session = coding_agent.create_session()

response = await coding_agent.run(
    "How does my AsyncWebScraper implementation align with Python's design principles?",
    session=session,
)
print("🤖 Agent:", response)

In [ ]:
response = await coding_agent.run(
    "Based on what you just said, when should I pick dataclasses versus Pydantic for this work?",
    session=session,
)
print("🤖 Agent:", response)
print("\n💡 The agent combined working memory (previous answer) with Cognee's knowledge graph.")

## Sesiune Nouă — Memoria pe Termen Lung Persistă

Începerea unei sesiuni noi șterge memoria de lucru, dar graficul de cunoștințe Cognee este încă disponibil. Agentul poate recupera aceeași cunoștință pe termen lung într-o conversație complet nouă.


In [ ]:
session_2 = coding_agent.create_session()

response = await coding_agent.run(
    "What logging guidance should I follow for incident reviews?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 New session, but the agent still has access to the full Cognee knowledge graph.")

In [ ]:
response = await coding_agent.run(
    "How should variables be named according to Python best practices?",
    session=session_2,
)
print("🤖 Agent:", response)

## Rezumat

În acest caiet ai construit un asistent de codare care combină **memoria de lucru a MAF** (`agent.create_session()`) cu **graful de cunoștințe pe termen lung al Cognee**.

### Ce ai învățat
1. **Construcția grafului de cunoștințe**: Cognee ingerează text neformatat și construiește un graf + memorie vectorială.
2. **Îmbogățirea grafului cu memify**: Fapte derivate și relații mai bogate deasupra grafului existent.
3. **Integrarea MAF + Cognee**: Funcțiile `@tool` permit unui agent MAF să interogheze graful Cognee în mod natural.
4. **Memorie de lucru + memorie pe termen lung**: `AgentSession` (prin `agent.create_session()`) oferă contextul sesiunii în timp ce Cognee oferă cunoștințe persistente.
5. **Căutare filtrată cu NodeSets**: Vizează subseturi specifice din graful de cunoștințe (de ex. doar principii).

### Concluzii cheie
- **Cognee** transformă textul brut în memorie structurată și conștientă de relații — mai puternică decât un magazin vectorial plat.
- **Funcțiile `@tool`** realizează o punte clară între agenții MAF și sistemele externe de cunoștințe.
- **`AgentSession`** (prin `agent.create_session()`) păstrează contextul per conversație separat de cunoștințele pe termen lung.
- Același graf de cunoștințe deservește multiple sesiuni și agenți.

### Aplicații din lumea reală
- **Copiloți pentru dezvoltatori**: Revizuire cod, analiza incidentelor, asistenți pentru arhitectură
- **Copiloți pentru clienți**: Agenți de suport cu acces la documentație produs, întrebări frecvente și note CRM
- **Copiloți experți interni**: Asistenți pentru politici, juridic sau securitate care raționează în baza ghidurilor
- **Straturi unificate de date**: Combină date structurate și nestructurate într-un graf interogabil

### Pașii următori
- Experimentează cu conștientizarea temporală în Cognee
- Definește o ontologie OWL pentru calitatea grafului specific domeniului
- Adaugă bucle de feedback utilizator pentru a îmbunătăți recuperarea în timp
- Scalează către sisteme multi-agent care împart aceeași memorie Cognee


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Declinare a responsabilității**:
Acest document a fost tradus folosind serviciul de traducere AI [Co-op Translator](https://github.com/Azure/co-op-translator). În timp ce ne străduim pentru acuratețe, vă rugăm să rețineți că traducerile automate pot conține erori sau inexactități. Documentul original în limba sa nativă trebuie considerat sursa autorizată. Pentru informații critice, se recomandă traducerea profesională realizată de un om. Nu ne asumăm responsabilitatea pentru eventualele neînțelegeri sau interpretări greșite care decurg din utilizarea acestei traduceri.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
